In [2]:
%pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 19.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from lightgbm import LGBMClassifier

train_path = pd.read_csv("train.csv")
test_path = pd.read_csv("test.csv")

target = "Irrigation_Need"

X_original = train_path.drop(columns=[target, "id"], errors="ignore")
y = train_path[target]

print(train_path.shape)
print(y.value_counts())
print(y.value_counts(normalize=True))

(630000, 21)
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64
Irrigation_Need
Low       0.587170
Medium    0.379483
High      0.033348
Name: proportion, dtype: float64


In [5]:
## Feature Engineering

def add_irrigation_features(df):
    df = df.copy()
    eps = 1e-6

    df["moisture_deficit"] = 100 - df["Soil_Moisture"]
    df["ph_distance_from_neutral"] = (df["Soil_pH"] - 7).abs()
    df["rainfall_log1p"] = np.log1p(df["Rainfall_mm"].clip(lower=0))
    df["previous_irrigation_log1p"] = np.log1p(df["Previous_Irrigation_mm"].clip(lower=0))
    df["field_area_log1p"] = np.log1p(df["Field_Area_hectare"].clip(lower=0))

    df["heat_dryness"] = df["Temperature_C"] / (df["Soil_Moisture"] + 1)
    df["sun_wind_heat"] = df["Sunlight_Hours"] * df["Wind_Speed_kmh"] * df["Temperature_C"]
    df["evaporation_pressure"] = df["sun_wind_heat"] / (df["Humidity"] + 1)
    df["rainfall_per_area"] = df["Rainfall_mm"] / (df["Field_Area_hectare"] + eps)
    df["previous_irrigation_per_area"] = df["Previous_Irrigation_mm"] / (df["Field_Area_hectare"] + eps)
    df["total_water_signal"] = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
    df["total_water_per_area"] = df["total_water_signal"] / (df["Field_Area_hectare"] + eps)
    df["organic_moisture"] = df["Organic_Carbon"] * df["Soil_Moisture"]
    df["conductivity_moisture"] = df["Electrical_Conductivity"] * df["Soil_Moisture"]

    df["crop_season"] = df["Crop_Type"].astype(str) + "_" + df["Season"].astype(str)
    df["soil_crop"] = df["Soil_Type"].astype(str) + "_" + df["Crop_Type"].astype(str)
    df["irrigation_water"] = df["Irrigation_Type"].astype(str) + "_" + df["Water_Source"].astype(str)
    df["region_season"] = df["Region"].astype(str) + "_" + df["Season"].astype(str)

    return df

X_fe = add_irrigation_features(X_original)

X_train_fe, X_valid_fe, y_train, y_valid = train_test_split(
    X_fe,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

X_train_original = X_original.loc[X_train_fe.index]
X_valid_original = X_original.loc[X_valid_fe.index]

print("Original feature count:", X_original.shape[1])
print("Engineered feature count:", X_fe.shape[1])

Original feature count: 19
Engineered feature count: 37


In [6]:
## Pipelines and Tuned Candidate Models
def make_ohe(sparse=True):
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse)

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else X

def make_preprocessor(X_frame, scale_numeric=False, dense_output=False):
    numeric_features = X_frame.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = X_frame.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    categorical_steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe(sparse=not dense_output)),
    ]

    return ColumnTransformer(
        [
            ("num", Pipeline(numeric_steps), numeric_features),
            ("cat", Pipeline(categorical_steps), categorical_features),
        ],
        sparse_threshold=0.0 if dense_output else 1.0,
    )

def build_candidate_models(X_frame):
    nb_preprocessor = make_preprocessor(X_frame, scale_numeric=False, dense_output=True)
    linear_preprocessor = make_preprocessor(X_frame, scale_numeric=True, dense_output=False)
    tree_preprocessor = make_preprocessor(X_frame, scale_numeric=False, dense_output=False)

    return {
        "GaussianNB var=1e-9": Pipeline([
            ("preprocessor", nb_preprocessor),
            ("to_dense", FunctionTransformer(to_dense)),
            ("model", GaussianNB(var_smoothing=1e-9)),
        ]),
        "GaussianNB var=1e-7": Pipeline([
            ("preprocessor", nb_preprocessor),
            ("to_dense", FunctionTransformer(to_dense)),
            ("model", GaussianNB(var_smoothing=1e-7)),
        ]),
        "LogReg balanced C=0.5": Pipeline([
            ("preprocessor", linear_preprocessor),
            ("model", LogisticRegression(max_iter=1500, C=0.5, class_weight="balanced", solver="saga", n_jobs=-1, random_state=42)),
        ]),
        "LogReg balanced C=1.0": Pipeline([
            ("preprocessor", linear_preprocessor),
            ("model", LogisticRegression(max_iter=1500, C=1.0, class_weight="balanced", solver="saga", n_jobs=-1, random_state=42)),
        ]),
        "RandomForest regularized": Pipeline([
            ("preprocessor", tree_preprocessor),
            ("model", RandomForestClassifier(n_estimators=250, max_depth=24, min_samples_leaf=3, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=42)),
        ]),
        "RandomForest deeper": Pipeline([
            ("preprocessor", tree_preprocessor),
            ("model", RandomForestClassifier(n_estimators=350, max_depth=None, min_samples_leaf=2, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=42)),
        ]),
        "LightGBM balanced": Pipeline([
            ("preprocessor", tree_preprocessor),
            ("model", LGBMClassifier(objective="multiclass", n_estimators=300, learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, colsample_bytree=0.8, class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1)),
        ]),
        "LightGBM regularized": Pipeline([
            ("preprocessor", tree_preprocessor),
            ("model", LGBMClassifier(objective="multiclass", n_estimators=500, learning_rate=0.03, num_leaves=127, min_child_samples=60, subsample=0.9, colsample_bytree=0.9, reg_alpha=1.0, reg_lambda=2.0, class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1)),
        ]),
    }

candidate_models = build_candidate_models(X_fe)

In [7]:
## Fit, Tune, and Compare Models

def metrics_dict(y_true, preds):
    return {
        "accuracy": accuracy_score(y_true, preds),
        "balanced_accuracy": balanced_accuracy_score(y_true, preds),
        "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        "high_precision": precision_score(y_true, preds, labels=["High"], average="macro", zero_division=0),
        "high_recall": recall_score(y_true, preds, labels=["High"], average="macro", zero_division=0),
        "high_f1": f1_score(y_true, preds, labels=["High"], average="macro", zero_division=0),
    }

fitted_models = {}
rows = []

for name, model in candidate_models.items():
    print("Training:", name)
    model.fit(X_train_fe, y_train)
    fitted_models[name] = model
    preds = model.predict(X_valid_fe)
    row = {"model": name}
    row.update(metrics_dict(y_valid, preds))
    rows.append(row)

model_results = pd.DataFrame(rows).sort_values("balanced_accuracy", ascending=False)
model_results


Training: GaussianNB var=1e-9
Training: GaussianNB var=1e-7
Training: LogReg balanced C=0.5


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training: LogReg balanced C=1.0


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training: RandomForest regularized
Training: RandomForest deeper
Training: LightGBM balanced


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training: LightGBM regularized


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,accuracy,balanced_accuracy,macro_f1,high_precision,high_recall,high_f1
6,LightGBM balanced,0.984341,0.970458,0.966810,0.916647,0.944788,0.930505
7,LightGBM regularized,0.985040,0.969679,0.970094,0.938480,0.940267,0.939372
4,RandomForest regularized,0.984944,0.965103,0.970261,0.955741,0.925036,0.940138
5,RandomForest deeper,0.985317,0.964121,0.970916,0.962687,0.920990,0.941377
2,LogReg balanced C=0.5,0.866151,0.878308,0.781211,0.447478,0.924560,0.603074
3,LogReg balanced C=1.0,0.866135,0.878297,0.781186,0.447426,0.924560,0.603027
0,GaussianNB var=1e-9,0.805976,0.786256,0.699730,0.352709,0.785578,0.486837
1,GaussianNB var=1e-7,0.766746,0.768543,0.674151,0.364120,0.830319,0.506239


In [8]:
## Test Whether Engineered Features Help

feature_test_model_name = "RandomForest regularized"

original_model = build_candidate_models(X_original)[feature_test_model_name]
engineered_model = build_candidate_models(X_fe)[feature_test_model_name]

original_model.fit(X_train_original, y_train)
engineered_model.fit(X_train_fe, y_train)

original_preds = original_model.predict(X_valid_original)
engineered_preds = engineered_model.predict(X_valid_fe)

feature_set_comparison = pd.DataFrame([
    {"feature_set": "original only", **metrics_dict(y_valid, original_preds)},
    {"feature_set": "original + engineered", **metrics_dict(y_valid, engineered_preds)},
])

feature_set_comparison

,feature_set,accuracy,balanced_accuracy,macro_f1,high_precision,high_recall,high_f1
0,original only,0.985881,0.965632,0.971505,0.959516,0.925036,0.941960
1,original + engineered,0.984944,0.965103,0.970261,0.955741,0.925036,0.940138


In [9]:
## Probability Averaging Ensemble

def model_classes(model):
    return model.classes_ if hasattr(model, "classes_") else model.named_steps["model"].classes_

def aligned_proba(model, X, class_order):
    probs = model.predict_proba(X)
    classes = model_classes(model)
    aligned = np.zeros((probs.shape[0], len(class_order)))

    for j, cls in enumerate(classes):
        aligned[:, np.where(class_order == cls)[0][0]] = probs[:, j]

    return aligned

top_model_names = model_results.head(3)["model"].tolist()
class_order = model_classes(fitted_models[top_model_names[0]])

valid_prob_stack = np.stack([
    aligned_proba(fitted_models[name], X_valid_fe, class_order)
    for name in top_model_names
])

equal_probs = valid_prob_stack.mean(axis=0)
equal_preds = class_order[np.argmax(equal_probs, axis=1)]

weights = model_results.set_index("model").loc[top_model_names, "balanced_accuracy"].to_numpy()
weights = weights / weights.sum()
weighted_probs = np.tensordot(weights, valid_prob_stack, axes=(0, 0))
weighted_preds = class_order[np.argmax(weighted_probs, axis=1)]

ensemble_results = pd.DataFrame([
    {"model": "equal_probability_ensemble", **metrics_dict(y_valid, equal_preds)},
    {"model": "weighted_probability_ensemble", **metrics_dict(y_valid, weighted_preds)},
])

pd.concat([model_results.head(5), ensemble_results], ignore_index=True).sort_values("balanced_accuracy", ascending=False)


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,accuracy,balanced_accuracy,macro_f1,high_precision,high_recall,high_f1
0,LightGBM balanced,0.984341,0.970458,0.966810,0.916647,0.944788,0.930505
1,LightGBM regularized,0.985040,0.969679,0.970094,0.938480,0.940267,0.939372
6,weighted_probability_ensemble,0.985262,0.969191,0.970459,0.941935,0.938125,0.940026
5,equal_probability_ensemble,0.985254,0.969112,0.970413,0.941922,0.937887,0.939900
2,RandomForest regularized,0.984944,0.965103,0.970261,0.955741,0.925036,0.940138
3,RandomForest deeper,0.985317,0.964121,0.970916,0.962687,0.920990,0.941377
4,LogReg balanced C=0.5,0.866151,0.878308,0.781211,0.447478,0.924560,0.603074


In [10]:
## Classification Reports

best_individual_name = model_results.iloc[0]["model"]

print("Best individual model:", best_individual_name)
print(classification_report(y_valid, fitted_models[best_individual_name].predict(X_valid_fe), zero_division=0))

best_ensemble_name = ensemble_results.sort_values("balanced_accuracy", ascending=False).iloc[0]["model"]
best_ensemble_preds = weighted_preds if best_ensemble_name == "weighted_probability_ensemble" else equal_preds

print("Best ensemble:", best_ensemble_name)
print(classification_report(y_valid, best_ensemble_preds, zero_division=0))


Best individual model: LightGBM balanced


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

        High       0.92      0.94      0.93      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.96      0.97      0.97    126000
weighted avg       0.98      0.98      0.98    126000

Best ensemble: weighted_probability_ensemble
              precision    recall  f1-score   support

        High       0.94      0.94      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.97      0.97      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [11]:
## Permutation Importance

importance_model = fitted_models[best_individual_name]

sample_n = min(10000, len(X_valid_fe))
sample_X = X_valid_fe.sample(sample_n, random_state=42)
sample_y = y_valid.loc[sample_X.index]

perm = permutation_importance(
    importance_model,
    sample_X,
    sample_y,
    scoring="balanced_accuracy",
    n_repeats=3,
    random_state=42,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_valid_fe.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.head(25)


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,feature,importance_mean,importance_std
11,Crop_Growth_Stage,0.287064,0.000515
16,Mulching_Used,0.171505,0.008428
5,Temperature_C,0.165198,0.004806
2,Soil_Moisture,0.135653,0.003843
9,Wind_Speed_kmh,0.130772,0.003680
7,Rainfall_mm,0.033164,0.001896
29,total_water_signal,0.012362,0.000552
19,moisture_deficit,0.009296,0.001494
6,Humidity,0.001993,0.001283
17,Previous_Irrigation_mm,0.001857,0.000504


In [13]:
## Kaggle Ensemble Submission

best_ensemble_name = ensemble_results.sort_values("balanced_accuracy", ascending=False).iloc[0]["model"]

final_models = {}
for name in top_model_names:
    print("Refitting on full train:", name)
    final_model = clone(candidate_models[name])
    final_model.fit(X_fe, y)
    final_models[name] = final_model

X_test_kaggle = add_irrigation_features(test_path.drop(columns=["id"], errors="ignore"))

test_prob_stack = np.stack([
    aligned_proba(final_models[name], X_test_kaggle, class_order)
    for name in top_model_names
])

if best_ensemble_name == "weighted_probability_ensemble":
    final_probs = np.tensordot(weights, test_prob_stack, axes=(0, 0))
else:
    final_probs = test_prob_stack.mean(axis=0)

final_preds = class_order[np.argmax(final_probs, axis=1)]

submission = pd.DataFrame({
    "id": test_path["id"],
    target: final_preds,
})

submission_path = "S6E4_hw4_ensemble_submission.csv"
submission.to_csv(submission_path, index=False)

submission.head()

Refitting on full train: LightGBM balanced
Refitting on full train: LightGBM regularized
Refitting on full train: RandomForest regularized


c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Owner\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


# Kaggle Playground Series S6E4 - Homework 4

This notebook section is for Homework 4. I focused on building several meaningfully different models, creating and testing new features, evaluating feature usefulness, and combining models with a probability averaging ensemble.

## Notebook Links

- Homework 4 notebook: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/hw4_playground_series.ipynb
- Original Homework document: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/KaggleHW-%20Sydney%20Thompson%20-%20sydneylienthompson.ipynb
- Initial EDA: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/initial_EDA.ipynb
- Random Forest: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/random_forest.ipynb
- XGBoost: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/xgboost_model.ipynb
- LightGBM: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/lightgbm_model.ipynb
- CatBoost: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/catboost_model.ipynb

## Approaches Tried

I trained and compared several models that differ in model type and tuning choices. Gaussian Naive Bayes was included as a simple probabilistic baseline. Logistic Regression was included as a regularized linear model with class weighting. Random Forest was included as a bagging/tree-based model, and LightGBM was included as a boosted tree model that usually performs well on tabular data.

The tuning choices were intentionally different across models. For Naive Bayes, I tested different `var_smoothing` values. For Logistic Regression, I tested different `C` values with balanced class weights. For Random Forest, I changed depth and leaf-size regularization. For LightGBM, I changed learning rate, number of trees, number of leaves, sampling, and regularization.

## Feature Engineering

I created new features to represent irrigation-related patterns: moisture deficit, pH distance from neutral, log transformations, water per field area, total water signal, heat/dryness, evaporation pressure, and categorical groupings such as `crop_season`, `soil_crop`, `irrigation_water`, and `region_season`. This increased the feature count from 19 original features to 37 engineered features.

The Random Forest feature-set comparison showed that the engineered feature set did not improve that model's validation score. The original-only Random Forest had balanced accuracy 0.965632 and High-class F1 0.941960, while the engineered-feature Random Forest had balanced accuracy 0.965103 and High-class F1 0.940138. This suggests that some engineered features may be redundant with the original variables, although they are still useful to test and interpret.

Permutation importance showed that the most useful features were `Crop_Growth_Stage`, `Mulching_Used`, `Temperature_C`, `Soil_Moisture`, and `Wind_Speed_kmh`. The engineered features `total_water_signal` and `moisture_deficit` also appeared in the top results, but their importance was much smaller than the strongest original features.

## Model and Ensemble Results

| Approach | Validation Balanced Accuracy | Macro F1 | High Class F1 | Kaggle LB Score |
|---|---:|---:|---:|---:|
| GaussianNB var=1e-9 | 0.786256 | 0.699730 | 0.486837 | not submitted |
| GaussianNB var=1e-7 | 0.768543 | 0.674151 | 0.506239 | not submitted |
| Logistic Regression C=0.5 | 0.878308 | 0.781211 | 0.603074 | not submitted |
| Logistic Regression C=1.0 | 0.878297 | 0.781186 | 0.603027 | not submitted |
| Random Forest regularized | 0.965103 | 0.970261 | 0.940138 | not submitted |
| Random Forest deeper | 0.964121 | 0.970916 | 0.941377 | not submitted |
| LightGBM balanced | 0.970458 | 0.966810 | 0.930505 | not submitted for HW4 |
| LightGBM regularized | 0.969679 | 0.970094 | 0.939372 | not submitted for HW4 |
| Equal probability ensemble | 0.969112 | 0.970413 | 0.939900 | not submitted |
| Weighted probability ensemble | 0.969191 | 0.970459 | 0.940026 | 0.96431 |
| Previous Random Forest submission | 0.9616 | not recorded | not recorded | 0.95986 |
| Previous XGBoost submission | 0.9685 | not recorded | not recorded | 0.95944 |
| Previous LightGBM submission | 0.9670 | not recorded | not recorded | 0.96833 |
| Previous CatBoost submission | 0.9710 | not recorded | not recorded | 0.96309 |

The best individual Homework 4 model by validation balanced accuracy was LightGBM balanced at 0.970458. The weighted probability ensemble reached 0.969191 balanced accuracy, so it did not improve over the best individual model on this validation split. However, the weighted ensemble had a stronger High-class F1 than the balanced LightGBM model, improving from 0.930505 to 0.940026. This is important because the High class is rare and practically important for irrigation decisions.

## What Worked and What Did Not

LightGBM worked best for balanced accuracy, which makes sense because this is a structured tabular problem with nonlinear relationships. Random Forest also performed well and gave strong High-class F1, especially the deeper version. Logistic Regression and Naive Bayes were much weaker, but they were still useful baselines because they showed how much the nonlinear tree-based models improved performance.

The engineered features did not clearly improve Random Forest performance in the direct original-vs-engineered comparison. The most important features were mostly original variables, especially crop growth stage, mulching, temperature, moisture, and wind speed. Moving forward, I would keep testing feature engineering but focus more on model tuning, thresholding for the High class, and ensembles or stacking with models that make different types of errors.

The ensemble improvement was mixed. It did not beat the best individual model on balanced accuracy, but it gave a competitive High-class F1. I would continue using probability averaging as a quick ensemble method, but I would also test stacking or custom class-specific thresholds if the goal is to improve the rare High class.